# Fast-dLLM v2 + Qwen block speculative decoding
Select a GPU runtime. Replace the repository URL and model paths below.

In [ ]:
!nvidia-smi
import torch, subprocess
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print('GPU:', torch.cuda.get_device_name())
    print(f'Memory free/total: {free/2**30:.2f}/{total/2**30:.2f} GiB')
    print('CUDA:', torch.version.cuda)

In [ ]:
REPOSITORY = 'https://github.com/<USER>/<REPOSITORY>.git'
BRANCH = 'feature/block-spec-fastdllm-qwen'
!git clone --branch $BRANCH $REPOSITORY
%cd <REPOSITORY>

In [ ]:
!pip install -r requirements-colab.txt
# PyTorch is intentionally not reinstalled.

In [ ]:
import os
os.environ['DRAFTER_MODEL_PATH'] = '/content/models/fast-dllm-1.5b'
os.environ['VERIFIER_MODEL_PATH'] = '/content/models/qwen-7b'
assert os.path.isdir(os.environ['DRAFTER_MODEL_PATH'])
assert os.path.isdir(os.environ['VERIFIER_MODEL_PATH'])

In [ ]:
!python scripts/inspect_models.py --drafter-model-path "$DRAFTER_MODEL_PATH" --verifier-model-path "$VERIFIER_MODEL_PATH"

In [ ]:
!python scripts/run_generation.py --config configs/colab_light.yaml --drafter-model-path "$DRAFTER_MODEL_PATH" --verifier-model-path "$VERIFIER_MODEL_PATH" --prompt "Solve: Natalia sold 48 clips in April and half as many in May." --max-new-tokens 16

In [ ]:
!python scripts/run_generation.py --config configs/colab_balanced.yaml --drafter-model-path "$DRAFTER_MODEL_PATH" --verifier-model-path "$VERIFIER_MODEL_PATH" --prompt "Solve: Natalia sold 48 clips in April and half as many in May." --max-new-tokens 128

In [ ]:
!python scripts/target_only.py --verifier-model-path "$VERIFIER_MODEL_PATH" --prompt "Solve: Natalia sold 48 clips in April and half as many in May." --max-new-tokens 128
!python scripts/benchmark.py
# Review the matrix, then execute on a long-lived GPU runtime with: !python scripts/benchmark.py --run